In [ ]:
# config bootstrap (auto-added): resolve repo paths from config.py
import os as _os, sys as _sys
_h = _os.path.abspath(_os.getcwd())
while not _os.path.exists(_os.path.join(_h, 'config.py')) and _os.path.dirname(_h) != _h:
    _h = _os.path.dirname(_h)
_sys.path.insert(0, _h)
import config as _cfg

# Image Authenticity Module — SightEngine API

Uses the SightEngine `genai` model to detect AI-generated images.
Uploads raw image files (Option 2) to `https://api.sightengine.com/1.0/check.json`.
Tests on 10 real news photos from the dataset + 4 AI-generated images.

In [2]:
import json
import os
import time

import requests
import pandas as pd
from PIL import Image
import matplotlib.pyplot as plt

# ── SightEngine credentials ──
API_USER   = "1409625234"
API_SECRET = "WFwczdygGNWodnQ2Mn565QqXfumU866Q"

assert API_USER and API_SECRET, "Set API_USER and API_SECRET above."
print(f"API user: {API_USER}")
print(f"API secret: {API_SECRET[:4]}...{API_SECRET[-4:]}")

API user: 1409625234
API secret: WFwc...866Q


In [3]:
# ── SightEngine detection function ──
SIGHTENGINE_URL = "https://api.sightengine.com/1.0/check.json"

def sightengine_ai_detection(image_path: str) -> dict:
    """
    Upload a local image to SightEngine's genai model.
    Returns the ai_generated score (0-1) and full response.
    """
    params = {
        "models": "genai",
        "api_user": API_USER,
        "api_secret": API_SECRET,
    }

    with open(image_path, "rb") as img_file:
        files = {"media": (os.path.basename(image_path), img_file)}
        response = requests.post(SIGHTENGINE_URL, files=files, data=params, timeout=30)

    response.raise_for_status()
    data = response.json()

    if data.get("status") != "success":
        raise RuntimeError(f"API error: {data}")

    ai_score = data.get("type", {}).get("ai_generated", None)

    return {
        "ai_generated": ai_score,
        "full_response": data,
    }

print("sightengine_ai_detection() ready.")

sightengine_ai_detection() ready.


In [4]:
# ── Gather test images: 10 real + 4 AI-generated ──
DATASET_ROOT = _os.path.join(str(_cfg.ROOT), 'datasets', 'dataset')
IMAGE_BASE   = os.path.join(DATASET_ROOT, "origin")
TEST_DIR     = _os.path.join(str(_cfg.ROOT), 'test_images')

ANNOTATIONS_PATH = os.path.join(DATASET_ROOT, "data", "NewsClipPings", "merged_balanced", "train.json")
METADATA_PATH    = os.path.join(DATASET_ROOT, "data", "NewsClipPings", "metadata", "train.json")

with open(ANNOTATIONS_PATH, "r", encoding="utf-8") as f:
    annotations = json.load(f)["annotations"]
with open(METADATA_PATH, "r", encoding="utf-8") as f:
    metadata = json.load(f)

# 10 unique real news images
real_samples = []
seen = set()
for ann in annotations:
    if len(real_samples) >= 10:
        break
    img_id = str(ann["image_id"])
    if img_id in seen or img_id not in metadata:
        continue
    rel = metadata[img_id]["image_path"].replace("visual_news/", "", 1)
    img_path = os.path.join(IMAGE_BASE, rel)
    if not os.path.isfile(img_path):
        continue
    seen.add(img_id)
    real_samples.append({
        "path": img_path,
        "label": "REAL",
        "name": os.path.basename(img_path),
        "caption": metadata[img_id]["caption"][:60],
    })

# 4 AI-generated images
ai_dir = os.path.join(TEST_DIR, "ai_generated")
ai_samples = sorted([
    {"path": os.path.join(ai_dir, f), "label": "AI-GEN", "name": f, "caption": "AI-generated face"}
    for f in os.listdir(ai_dir) if f.endswith((".jpg", ".png"))
], key=lambda x: x["name"])

all_samples = real_samples + ai_samples
print(f"Real: {len(real_samples)} | AI-generated: {len(ai_samples)} | Total: {len(all_samples)}")
for s in all_samples:
    print(f"  [{s['label']:6s}] {s['name']}")

Real: 10 | AI-generated: 4 | Total: 14
  [REAL  ] 482.jpg
  [REAL  ] 113.jpg
  [REAL  ] 584.jpg
  [REAL  ] 036.jpg
  [REAL  ] 675.jpg
  [REAL  ] 703.jpg
  [REAL  ] 903.jpg
  [REAL  ] 979.jpg
  [REAL  ] 581.jpg
  [REAL  ] 320.jpg
  [AI-GEN] ai_face_1.jpg
  [AI-GEN] ai_face_2.jpg
  [AI-GEN] ai_face_3.jpg
  [AI-GEN] ai_face_4.jpg


In [5]:
# ── Run SightEngine on all images ──
results = []

for i, s in enumerate(all_samples):
    print(f"[{i+1}/{len(all_samples)}] {s['label']:6s} | {s['name']}...", end=" ")

    try:
        result = sightengine_ai_detection(s["path"])
        ai_score = result["ai_generated"]
        verdict = "AI-GENERATED" if ai_score is not None and ai_score > 0.5 else "REAL"
        correct = (s["label"] == "REAL" and verdict == "REAL") or \
                  (s["label"] == "AI-GEN" and verdict == "AI-GENERATED")

        results.append({
            "filename": s["name"],
            "ground_truth": s["label"],
            "ai_generated_score": ai_score,
            "verdict": verdict,
            "correct": correct,
            "caption": s["caption"],
            "path": s["path"],
        })
        mark = "Y" if correct else "X"
        print(f"AI={ai_score:.4f} | {verdict} [{mark}]")

    except Exception as e:
        print(f"ERROR: {e}")
        results.append({
            "filename": s["name"],
            "ground_truth": s["label"],
            "ai_generated_score": None,
            "verdict": "ERROR",
            "correct": False,
            "caption": s["caption"],
            "path": s["path"],
        })

    time.sleep(0.5)

print("\nDone!")

[1/14] REAL   | 482.jpg... AI=0.0100 | REAL [Y]
[2/14] REAL   | 113.jpg... AI=0.0100 | REAL [Y]
[3/14] REAL   | 584.jpg... AI=0.0100 | REAL [Y]
[4/14] REAL   | 036.jpg... AI=0.0100 | REAL [Y]
[5/14] REAL   | 675.jpg... AI=0.0100 | REAL [Y]
[6/14] REAL   | 703.jpg... AI=0.0100 | REAL [Y]
[7/14] REAL   | 903.jpg... AI=0.3100 | REAL [Y]
[8/14] REAL   | 979.jpg... AI=0.0100 | REAL [Y]
[9/14] REAL   | 581.jpg... AI=0.0100 | REAL [Y]
[10/14] REAL   | 320.jpg... AI=0.0100 | REAL [Y]
[11/14] AI-GEN | ai_face_1.jpg... AI=0.9900 | AI-GENERATED [Y]
[12/14] AI-GEN | ai_face_2.jpg... AI=0.9800 | AI-GENERATED [Y]
[13/14] AI-GEN | ai_face_3.jpg... AI=0.9900 | AI-GENERATED [Y]
[14/14] AI-GEN | ai_face_4.jpg... AI=0.9900 | AI-GENERATED [Y]

Done!


In [6]:
# ── Results table ──
df = pd.DataFrame(results)
display(df[["filename", "ground_truth", "ai_generated_score", "verdict", "correct", "caption"]])

valid = df[df["ai_generated_score"].notna()]
acc = valid["correct"].mean()

real_rows = valid[valid["ground_truth"] == "REAL"]
ai_rows = valid[valid["ground_truth"] == "AI-GEN"]

print(f"\n{'=' * 60}")
print(f"SIGHTENGINE AI-GENERATED DETECTION RESULTS")
print(f"{'=' * 60}")
print(f"  Accuracy: {acc:.0%} ({valid['correct'].sum()}/{len(valid)})")
print(f"\n  REAL news photos  — avg AI score: {real_rows['ai_generated_score'].mean():.4f}")
print(f"  AI-generated      — avg AI score: {ai_rows['ai_generated_score'].mean():.4f}")
print(f"  Separation        : {ai_rows['ai_generated_score'].mean() - real_rows['ai_generated_score'].mean():.4f}")

,filename,ground_truth,ai_generated_score,verdict,correct,caption
0,482.jpg,REAL,0.01,REAL,True,Saudi troops cheer as they ride at the back of...
1,113.jpg,REAL,0.01,REAL,True,Israeli soldiers ride on a tank to a position ...
2,584.jpg,REAL,0.01,REAL,True,Tiffany is north east of Aberdeen
3,036.jpg,REAL,0.01,REAL,True,Some ferry services were affected between Holy...
4,675.jpg,REAL,0.01,REAL,True,Rock band Paramore is another group which sell...
5,703.jpg,REAL,0.01,REAL,True,Hot air balloons lift off on the first day of ...
6,903.jpg,REAL,0.31,REAL,True,Independent Afghan civil society activist wome...
7,979.jpg,REAL,0.01,REAL,True,An Egyptian grave from 3400BC in the British M...
8,581.jpg,REAL,0.01,REAL,True,Kansas State Troopers stand outside a police l...
9,320.jpg,REAL,0.01,REAL,True,The Fraternal Order of Police s president said...



SIGHTENGINE AI-GENERATED DETECTION RESULTS
  Accuracy: 100% (14/14)

  REAL news photos  — avg AI score: 0.0400
  AI-generated      — avg AI score: 0.9875
  Separation        : 0.9475


In [ ]:
# ── Bar chart ──
fig, ax = plt.subplots(figsize=(12, 5))

labels = df["filename"].tolist()
scores = [s if s is not None else 0 for s in df["ai_generated_score"].tolist()]
colors = ["#2ecc71" if gt == "REAL" else "#e74c3c" for gt in df["ground_truth"].tolist()]
edge_colors = ["green" if c else "red" for c in df["correct"].tolist()]

bars = ax.bar(range(len(labels)), scores, color=colors, edgecolor=edge_colors, linewidth=2, alpha=0.85)
ax.axhline(y=0.5, color="orange", linestyle="--", linewidth=2, label="Decision boundary (0.5)")

ax.set_xticks(range(len(labels)))
ax.set_xticklabels(labels, rotation=45, ha="right", fontsize=8)
ax.set_ylabel("AI-Generated Score")
ax.set_title(f"SightEngine genai Scores — {acc:.0%} Accuracy")
ax.set_ylim(0, 1.05)
ax.legend()

plt.tight_layout()
plt.savefig("sightengine_results.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# ── Visual grid ──
n_real = len(real_rows)
n_ai = len(ai_rows)
n_cols = max(n_real, n_ai)

fig, axes = plt.subplots(2, n_cols, figsize=(4 * n_cols, 8))

real_results = [r for r in results if r["ground_truth"] == "REAL"]
ai_results = [r for r in results if r["ground_truth"] == "AI-GEN"]

for row_idx, (items, row_label) in enumerate([(real_results, "REAL"), (ai_results, "AI-GEN")]):
    for col_idx, item in enumerate(items):
        ax = axes[row_idx, col_idx] if n_cols > 1 else axes[row_idx]
        img = Image.open(item["path"]).convert("RGB")
        ax.imshow(img)

        ai_s = item["ai_generated_score"]
        color = "green" if item["correct"] else "red"
        score_str = f"{ai_s:.4f}" if ai_s is not None else "ERR"

        ax.set_title(f"GT: {item['ground_truth']} | {item['verdict']}\nAI: {score_str}",
                     fontsize=9, color=color, fontweight="bold")
        ax.tick_params(left=False, bottom=False, labelleft=False, labelbottom=False)
        for spine in ax.spines.values():
            spine.set_edgecolor(color)
            spine.set_linewidth(3)

    for col_idx in range(len(items), n_cols):
        ax = axes[row_idx, col_idx] if n_cols > 1 else axes[row_idx]
        ax.axis("off")

plt.suptitle("SightEngine: Real News (top) vs AI-Generated (bottom)", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()